# Phase 3 — NLP & Transformers
## Day 11: Text Preprocessing & TF-IDF
**Date:** 2026-04-24

### Learning Objectives
- Tokenize text into words and sentences
- Remove stopwords and apply stemming/lemmatization
- Build document-term matrices with CountVectorizer
- Understand TF-IDF and use TfidfVectorizer
- Compare raw counts vs TF-IDF for text classification

In [ ]:
# Setup - install if needed
# !pip install nltk scikit-learn pandas numpy

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print('All imports and downloads ready!')

In [ ]:
# Sample data: product reviews with sentiment labels
reviews = [
    "This phone is absolutely amazing! Best purchase ever.",
    "Terrible battery life. The screen broke after one week.",
    "Great value for the money. Camera quality is superb.",
    "Worst product I have ever bought. Total waste of money.",
    "Love the design and the fast processor. Highly recommend!",
    "The software keeps crashing. Very frustrating experience.",
    "Excellent build quality and beautiful display.",
    "Cheap plastic feel. Overpriced for what you get.",
    "Runs smoothly and the battery lasts all day long.",
    "Disappointing camera. Photos look blurry and washed out.",
    "Perfect for everyday use. Very happy with this purchase.",
    "Returned it after two days. Constant freezing issues.",
    "Lightweight and portable. Great for travel.",
    "Overheats quickly and the fan is extremely loud.",
    "Sleek design with amazing sound quality. Love it!",
    "Customer support was unhelpful. Product arrived damaged.",
    "Fast shipping and the product exceeded my expectations.",
    "Screen resolution is terrible. Colors look faded.",
    "Best laptop I have owned. Worth every penny.",
    "Stopped working after a month. Very poor quality.",
]
labels = [1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0]  # 1=positive, 0=negative

df = pd.DataFrame({'review': reviews, 'sentiment': labels})
print(f"Dataset: {len(df)} reviews, {df.sentiment.sum()} positive, {(df.sentiment==0).sum()} negative")
df.head()

## 1. Tokenization

Tokenization splits text into smaller pieces (tokens). You can split into words or sentences. NLTK provides `word_tokenize` and `sent_tokenize` for this.

Why does it matter? Machines don't understand raw text. We need to break it into units we can count, filter, and transform.

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize

sample = "This phone is absolutely amazing! Best purchase ever. I'd buy it again."

# Word tokenization
words = word_tokenize(sample)
print("Word tokens:", words)
print(f"Number of tokens: {len(words)}")

# Sentence tokenization
sentences = sent_tokenize(sample)
print("\nSentence tokens:", sentences)
print(f"Number of sentences: {len(sentences)}")

In [ ]:
# Simple tokenization with .split() vs nltk
text = "I'd like to buy it. Can't wait!"

print("split():", text.split())
print("nltk:   ", word_tokenize(text))
# Notice how nltk handles contractions like "I'd" -> ["I", "'d"]

## 2. Stopwords

Stopwords are common words like "the", "is", "and" that carry little meaning on their own. Removing them reduces noise and makes our features more meaningful.

NLTK has built-in stopword lists for many languages.

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
print(f"Number of English stopwords: {len(stop_words)}")
print(f"Some examples: {sorted(list(stop_words))[:15]}")

# Remove stopwords from a sentence
sample = "This phone is absolutely amazing and the best purchase ever"
tokens = word_tokenize(sample.lower())
filtered = [w for w in tokens if w not in stop_words and w.isalpha()]

print(f"\nBefore: {tokens}")
print(f"After:  {filtered}")

## 3. Stemming vs Lemmatization

Both reduce words to their base form, but they work differently:

**Stemming** chops off word endings with simple rules. It's fast but can produce non-real words ("studies" -> "studi").

**Lemmatization** uses a dictionary to find the actual root word ("studies" -> "study"). It's slower but produces real words.

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

test_words = ['running', 'studies', 'better', 'cats', 'flying', 'happily', 'organization']

print(f"{'Word':<15} {'Stemmed':<15} {'Lemmatized':<15}")
print('-' * 45)
for word in test_words:
    print(f"{word:<15} {stemmer.stem(word):<15} {lemmatizer.lemmatize(word):<15}")

In [ ]:
# Lemmatizer with POS tag gives better results
# By default it assumes nouns. Tell it about verbs:
print("lemmatize('running'):         ", lemmatizer.lemmatize('running'))          # noun assumption
print("lemmatize('running', 'v'):     ", lemmatizer.lemmatize('running', 'v'))    # verb -> 'run'
print("lemmatize('better', 'a'):      ", lemmatizer.lemmatize('better', 'a'))     # adjective -> 'good'

## 4. Full Preprocessing Pipeline

Let's combine everything into a reusable function: lowercase, tokenize, remove stopwords and punctuation, then lemmatize.

In [ ]:
def preprocess(text):
    """Full text preprocessing pipeline."""
    # Lowercase
    text = text.lower()
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and non-alpha tokens
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    # Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Apply to our dataset
df['clean'] = df['review'].apply(preprocess)
print("Original:", df['review'].iloc[0])
print("Cleaned: ", df['clean'].iloc[0])
print()
df[['review', 'clean']].head()

## 5. CountVectorizer (Bag of Words)

CountVectorizer converts text into a matrix where each column is a word and each cell is the count of that word in that document. This is the simplest way to turn text into numbers.

The result is a "document-term matrix" (DTM). Each row = one document. Each column = one unique word.

In [ ]:
# Simple example first
tiny_docs = [
    "I love cats",
    "I love dogs",
    "cats and dogs are great"
]

cv = CountVectorizer()
dtm = cv.fit_transform(tiny_docs)

print("Vocabulary:", cv.get_feature_names_out())
print("\nDocument-Term Matrix:")
print(pd.DataFrame(dtm.toarray(), columns=cv.get_feature_names_out()))

In [ ]:
# CountVectorizer on our reviews
cv = CountVectorizer(max_features=50)  # keep top 50 words
X_counts = cv.fit_transform(df['clean'])

print(f"Matrix shape: {X_counts.shape}  (documents x features)")
print(f"Vocabulary size: {len(cv.get_feature_names_out())}")
print(f"Top features: {cv.get_feature_names_out()[:10]}")

# The matrix is sparse (mostly zeros)
print(f"\nNon-zero entries: {X_counts.nnz} out of {X_counts.shape[0] * X_counts.shape[1]}")

## 6. TF-IDF: Term Frequency - Inverse Document Frequency

The problem with raw counts: common words get high scores even if they appear in every document and aren't useful for distinguishing between documents.

TF-IDF fixes this by downweighting words that appear in many documents:
- **TF** (Term Frequency) = how often a word appears in a document
- **IDF** (Inverse Document Frequency) = log(total docs / docs containing the word). Rare words get high IDF, common words get low IDF.
- **TF-IDF** = TF x IDF

A word that appears often in one document but rarely across all documents gets a high TF-IDF score. That's usually a meaningful word.

In [ ]:
# TF-IDF on our tiny example
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(tiny_docs)

print("TF-IDF Matrix (compare with raw counts above):")
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(3),
    columns=tfidf.get_feature_names_out()
)
print(tfidf_df)
print()
# Notice: 'love' and 'i' appear in docs 0 and 1 -> lower scores
# 'cats' appears in docs 0 and 2, 'dogs' in docs 1 and 2
# 'great' only appears in doc 2 -> higher score

In [ ]:
# TF-IDF on our reviews dataset
tfidf_vec = TfidfVectorizer(max_features=50)
X_tfidf = tfidf_vec.fit_transform(df['clean'])

print(f"TF-IDF matrix shape: {X_tfidf.shape}")

# Show top TF-IDF words for the first positive and first negative review
feature_names = tfidf_vec.get_feature_names_out()
for idx, label in [(0, 'Positive'), (1, 'Negative')]:
    row = X_tfidf[idx].toarray().flatten()
    top_indices = row.argsort()[-5:][::-1]
    top_words = [(feature_names[i], round(row[i], 3)) for i in top_indices if row[i] > 0]
    print(f"\n{label} review: \"{df['review'].iloc[idx][:50]}...\"")
    print(f"  Top TF-IDF words: {top_words}")

## 7. N-grams

So far we've looked at single words (unigrams). But sometimes word pairs or triples carry more meaning. "not good" means the opposite of "good"!

Both CountVectorizer and TfidfVectorizer support n-grams via the `ngram_range` parameter.

In [ ]:
# Bigrams (2-word combinations)
bigram_vec = CountVectorizer(ngram_range=(1, 2), max_features=30)
X_bigram = bigram_vec.fit_transform(df['clean'])

print("Features with bigrams:")
print(bigram_vec.get_feature_names_out())
print(f"\nShape: {X_bigram.shape} (more features because of word pairs)")

## 8. Counts vs TF-IDF for Classification

Let's compare how raw counts and TF-IDF perform when we use them to classify our reviews as positive or negative.

In [ ]:
from sklearn.model_selection import cross_val_score

# Prepare features
cv_full = CountVectorizer(max_features=100)
tfidf_full = TfidfVectorizer(max_features=100)

X_cv = cv_full.fit_transform(df['clean'])
X_tf = tfidf_full.fit_transform(df['clean'])
y = df['sentiment']

# Compare with cross-validation
lr = LogisticRegression(max_iter=1000)

scores_cv = cross_val_score(lr, X_cv, y, cv=3, scoring='accuracy')
scores_tf = cross_val_score(lr, X_tf, y, cv=3, scoring='accuracy')

print(f"CountVectorizer accuracy: {scores_cv.mean():.3f} (+/- {scores_cv.std():.3f})")
print(f"TfidfVectorizer accuracy: {scores_tf.mean():.3f} (+/- {scores_tf.std():.3f})")
print("\nWith small datasets, results can be similar. TF-IDF usually shines with larger corpora.")

## 9. Useful Parameters

Both vectorizers have handy parameters you should know:

In [ ]:
# Key parameters demo
vec = TfidfVectorizer(
    max_features=50,        # keep only top N features
    min_df=2,               # ignore words appearing in fewer than 2 docs
    max_df=0.9,             # ignore words appearing in more than 90% of docs
    ngram_range=(1, 2),     # unigrams and bigrams
    stop_words='english',   # built-in English stopwords (alternative to NLTK)
    sublinear_tf=True,      # apply log to TF (reduces impact of very frequent terms)
)

X_demo = vec.fit_transform(df['review'])  # using raw reviews, let sklearn handle stopwords
print(f"Shape: {X_demo.shape}")
print(f"Features: {vec.get_feature_names_out()[:15]}")
print("\nNote: sklearn's built-in stop_words='english' means you can skip manual preprocessing!")

---
## Tricky Bits

Common mistakes people make with text preprocessing and vectorization.

In [ ]:
# MISTAKE 1: Fitting the vectorizer on test data (data leakage!)
print("=== Mistake 1: Don't fit_transform on test data ===")
train_docs = ["I love this product", "Terrible experience"]
test_docs = ["I love the experience"]

vec = CountVectorizer()

# WRONG: fitting on test data too
X_train_wrong = vec.fit_transform(train_docs)
X_test_wrong = vec.fit_transform(test_docs)  # This refits! Different vocabulary!
print(f"Train features: {vec.get_feature_names_out()}")
print(f"Train shape: {X_train_wrong.shape}, Test shape: {X_test_wrong.shape}")
print("Shapes don't match! This will crash your model.\n")

# RIGHT: fit on train, transform on test
vec2 = CountVectorizer()
X_train_right = vec2.fit_transform(train_docs)
X_test_right = vec2.transform(test_docs)  # just transform, don't fit
print(f"Features: {vec2.get_feature_names_out()}")
print(f"Train shape: {X_train_right.shape}, Test shape: {X_test_right.shape}")
print("Shapes match! 'the' in test gets ignored (not in train vocab).")

In [ ]:
# MISTAKE 2: Forgetting that vectorizer output is sparse
print("=== Mistake 2: Sparse vs Dense ===")
vec = CountVectorizer()
X = vec.fit_transform(["hello world", "world cup"])
print(f"Type: {type(X)}")
print(f"Trying to index like a numpy array...")
try:
    print(X[0, 0])  # This works
    print("Single indexing works on sparse matrices.")
except Exception as e:
    print(f"Error: {e}")

# To use with pandas or see all values, convert to dense:
print(f"\nDense version:\n{X.toarray()}")
print("Use .toarray() when you need a regular numpy array.")

In [ ]:
# MISTAKE 3: Over-preprocessing (removing meaningful words)
print("=== Mistake 3: Stopword removal can hurt ===")
sentences = ["not good at all", "very good indeed"]

# With stopwords removed, both look the same!
vec_stop = TfidfVectorizer(stop_words='english')
X = vec_stop.fit_transform(sentences)
print(f"With stopwords removed: {vec_stop.get_feature_names_out()}")
print(f"Both sentences produce the same features! 'not' was removed.")
print("\nSolution: use bigrams so 'not good' is captured as one feature.")

vec_bi = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
X2 = vec_bi.fit_transform(sentences)
print(f"With bigrams: {vec_bi.get_feature_names_out()}")

---
## Trick Questions

Test your understanding. Try to answer before revealing the solution.

**Q1:** If a word appears in every single document, what is its IDF score?
<details><summary>Answer</summary>
Close to zero. IDF = log(N / df), where df = N (all docs). log(1) = 0. Sklearn adds 1 to avoid zero: log((1+N)/(1+df)) + 1, so it's still very low.
</details>

**Q2:** You have 1000 documents and set `max_features=100`. Does the vectorizer pick the 100 most common words or the 100 with highest TF-IDF?
<details><summary>Answer</summary>
For CountVectorizer: the 100 most frequent words by total count across the corpus. For TfidfVectorizer: the 100 words with highest total TF-IDF score summed across all documents.
</details>

**Q3:** Why should you NOT use `fit_transform()` on your test set?
<details><summary>Answer</summary>
Because fit_transform learns a new vocabulary from the test data, which causes data leakage and mismatched feature dimensions. Always fit on train, then use transform() on test.
</details>

**Q4:** What's the difference between `stop_words='english'` in sklearn and using NLTK stopwords?
<details><summary>Answer</summary>
They use different stopword lists. Sklearn's list has about 318 words, NLTK's has about 179. Sklearn's is more aggressive. You can also pass a custom list to either.
</details>

**Q5:** If you stem "organization", "organize", and "organ", do they all become the same token?
<details><summary>Answer</summary>
With PorterStemmer: "organization" -> "organ", "organize" -> "organ", "organ" -> "organ". Yes, they all collapse to "organ"! This is a known downside of aggressive stemming.
</details>

---
## Exercises

Fill in the `___` blanks. Each cell has asserts to check your work.

In [ ]:
# Exercise 1: Tokenize this sentence into words using nltk
from nltk.tokenize import word_tokenize

sentence = "Machine learning is fascinating!"
tokens = ___(sentence)

assert isinstance(tokens, list)
assert 'Machine' in tokens
assert '!' in tokens
print(f"Tokens: {tokens}")
print("Exercise 1 passed!")

In [ ]:
# Exercise 2: Remove stopwords from a list of tokens
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

tokens = ['the', 'cat', 'is', 'sitting', 'on', 'the', 'mat']
filtered = [w for w in tokens if w not in ___]

assert 'the' not in filtered
assert 'is' not in filtered
assert 'cat' in filtered
assert 'sitting' in filtered
print(f"Filtered: {filtered}")
print("Exercise 2 passed!")

In [ ]:
# Exercise 3: Lemmatize a verb correctly
from nltk.stem import WordNetLemmatizer
lem = WordNetLemmatizer()

# Hint: you need to pass the POS tag for verbs
result = lem.lemmatize('running', ___)

assert result == 'run'
print(f"'running' -> '{result}'")
print("Exercise 3 passed!")

In [ ]:
# Exercise 4: Create a TfidfVectorizer with bigrams and max 20 features
from sklearn.feature_extraction.text import TfidfVectorizer

docs = ["I love machine learning", "deep learning is great", "machine learning and deep learning"]

vec = TfidfVectorizer(ngram_range=___, max_features=___)
X = vec.fit_transform(docs)

assert X.shape[1] <= 20
# Check that bigrams are present
features = list(vec.get_feature_names_out())
assert any(' ' in f for f in features), "No bigrams found!"
print(f"Features: {features}")
print(f"Shape: {X.shape}")
print("Exercise 4 passed!")

In [ ]:
# Exercise 5: Fit on train, transform test (no data leakage)
from sklearn.feature_extraction.text import CountVectorizer

train = ["good movie", "bad movie", "great film"]
test = ["good film"]

vec = CountVectorizer()
X_train = vec.___(train)       # fit + transform on train
X_test = vec.___(test)          # only transform on test

assert X_train.shape[1] == X_test.shape[1], "Feature dimensions must match!"
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Vocabulary: {vec.get_feature_names_out()}")
print("Exercise 5 passed!")

In [ ]:
# Exercise 6: Convert sparse matrix to dense numpy array
vec = CountVectorizer()
X_sparse = vec.fit_transform(["hello world", "world peace"])

X_dense = X_sparse.___()

assert isinstance(X_dense, np.ndarray)
assert X_dense.shape == X_sparse.shape
print(f"Dense array:\n{X_dense}")
print(f"Features: {vec.get_feature_names_out()}")
print("Exercise 6 passed!")

In [ ]:
# Exercise 7: Build a complete preprocessing function
# Combine: lowercase, tokenize, remove stopwords, lemmatize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop = set(stopwords.words('english'))
lem = WordNetLemmatizer()

def clean_text(text):
    text = text.___()                                    # lowercase
    tokens = ___(text)                                   # tokenize
    tokens = [t for t in tokens if t.isalpha() and t not in ___]  # filter
    tokens = [lem.___(t) for t in tokens]                # lemmatize
    return ' '.join(tokens)

result = clean_text("The cats are running quickly towards the Houses!")
assert 'the' not in result
assert 'cat' in result
assert 'running' in result or 'run' in result  # lemmatize may or may not catch without POS
print(f"Result: '{result}'")
print("Exercise 7 passed!")

### Solutions

<details><summary>Exercise 1</summary>

```python
tokens = word_tokenize(sentence)
```
</details>

<details><summary>Exercise 2</summary>

```python
filtered = [w for w in tokens if w not in stop_words]
```
</details>

<details><summary>Exercise 3</summary>

```python
result = lem.lemmatize('running', 'v')
```
</details>

<details><summary>Exercise 4</summary>

```python
vec = TfidfVectorizer(ngram_range=(1, 2), max_features=20)
```
</details>

<details><summary>Exercise 5</summary>

```python
X_train = vec.fit_transform(train)
X_test = vec.transform(test)
```
</details>

<details><summary>Exercise 6</summary>

```python
X_dense = X_sparse.toarray()
```
</details>

<details><summary>Exercise 7</summary>

```python
def clean_text(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha() and t not in stop]
    tokens = [lem.lemmatize(t) for t in tokens]
    return ' '.join(tokens)
```
</details>

---
## Cumulative Review Exercises

Mixed exercises covering Days 1-10. Fill in the blanks and run the asserts.

In [ ]:
# Review 1 (Day 1 - Pandas): Filter rows where a column value meets a condition
import pandas as pd

df_r = pd.DataFrame({'name': ['Alice', 'Bob', 'Charlie'], 'score': [85, 92, 78]})
high_scorers = df_r[df_r['score'] > ___]  # Get scores above 80

assert len(high_scorers) == 2
assert 'Charlie' not in high_scorers['name'].values
print("Review 1 passed!")

In [ ]:
# Review 2 (Day 2 - Numpy): Create a 3x3 array and compute column means
import numpy as np

arr = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
col_means = arr.mean(axis=___)

assert col_means.shape == (3,)
assert col_means[0] == 4.0
print(f"Column means: {col_means}")
print("Review 2 passed!")

In [ ]:
# Review 3 (Day 3 - Data Cleaning): Fill missing values with the column mean
import pandas as pd
import numpy as np

df_r3 = pd.DataFrame({'val': [10, np.nan, 30, np.nan, 50]})
df_r3['val'] = df_r3['val'].___(df_r3['val'].mean())

assert df_r3['val'].isna().sum() == 0
assert df_r3['val'].iloc[1] == 30.0
print("Review 3 passed!")

In [ ]:
# Review 4 (Day 4 - Python Core): Write a list comprehension that squares even numbers
numbers = [1, 2, 3, 4, 5, 6]
squared_evens = [x**2 for x in numbers if x % ___ == 0]

assert squared_evens == [4, 16, 36]
print("Review 4 passed!")

In [ ]:
# Review 5 (Day 6 - Pipelines): Create a train/test split with 20% test size
from sklearn.model_selection import train_test_split
import numpy as np

X = np.arange(100).reshape(50, 2)
y = np.random.randint(0, 2, 50)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=___, random_state=42)

assert len(X_test) == 10
assert len(X_train) == 40
print("Review 5 passed!")

In [ ]:
# Review 6 (Day 7 - LogReg): Fit a logistic regression and get predictions
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=100, n_features=5, random_state=42)
model = LogisticRegression(max_iter=1000)
model.___(X, y)
preds = model.___(X)

assert len(preds) == 100
assert set(preds).issubset({0, 1})
print(f"Accuracy: {(preds == y).mean():.2f}")
print("Review 6 passed!")

In [ ]:
# Review 7 (Day 8 - Ensembles): Import and create a Random Forest classifier
from sklearn.ensemble import ___

rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X, y)
print(f"Number of trees: {len(rf.estimators_)}")
assert len(rf.estimators_) == 50
print("Review 7 passed!")

In [ ]:
# Review 8 (Day 9 - Metrics): Compute precision from a confusion matrix
from sklearn.metrics import confusion_matrix, precision_score

y_true = [1, 0, 1, 1, 0, 1, 0, 0, 1, 1]
y_pred = [1, 0, 1, 0, 0, 1, 1, 0, 1, 1]

precision = precision_score(y_true, y_pred)
# Precision = TP / (TP + FP)
# TP = 5, FP = 1 -> precision = 5/6

assert round(precision, 3) == round(5/6, 3)
print(f"Precision: {precision:.3f}")
print("Review 8 passed!")

In [ ]:
# Review 9 (Day 9 - Metrics): What does ROC-AUC measure? Fill in the string.
answer = "___"  # Replace with: "threshold-independent" or "threshold-dependent"

assert answer.lower() == "threshold-independent"
print("ROC-AUC is threshold-independent: it measures performance across ALL thresholds.")
print("Review 9 passed!")

In [ ]:
# Review 10 (Day 10 - SHAP): What does a positive SHAP value mean for a feature?
answer = "___"  # Replace with: "pushes prediction higher" or "pushes prediction lower"

assert answer.lower() == "pushes prediction higher"
print("Positive SHAP value = that feature pushed the prediction higher for this instance.")
print("Review 10 passed!")

### Cumulative Review Solutions

<details><summary>Review 1</summary>

```python
high_scorers = df_r[df_r['score'] > 80]
```
</details>

<details><summary>Review 2</summary>

```python
col_means = arr.mean(axis=0)
```
</details>

<details><summary>Review 3</summary>

```python
df_r3['val'] = df_r3['val'].fillna(df_r3['val'].mean())
```
</details>

<details><summary>Review 4</summary>

```python
squared_evens = [x**2 for x in numbers if x % 2 == 0]
```
</details>

<details><summary>Review 5</summary>

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```
</details>

<details><summary>Review 6</summary>

```python
model.fit(X, y)
preds = model.predict(X)
```
</details>

<details><summary>Review 7</summary>

```python
from sklearn.ensemble import RandomForestClassifier
```
</details>

<details><summary>Review 8</summary>

No blanks to fill. Just run and verify the precision is 5/6 ≈ 0.833.
</details>

<details><summary>Review 9</summary>

```python
answer = "threshold-independent"
```
</details>

<details><summary>Review 10</summary>

```python
answer = "pushes prediction higher"
```
</details>

In [ ]:
# Cheat Sheet: Text Preprocessing & TF-IDF
cheat_sheet = """
============================================
  TEXT PREPROCESSING & TF-IDF CHEAT SHEET
============================================

TOKENIZATION
  from nltk.tokenize import word_tokenize, sent_tokenize
  words = word_tokenize("Hello world!")     # ['Hello', 'world', '!']
  sents = sent_tokenize("Hi. Bye.")         # ['Hi.', 'Bye.']

STOPWORDS
  from nltk.corpus import stopwords
  stop = set(stopwords.words('english'))     # ~179 words
  filtered = [w for w in tokens if w not in stop]

STEMMING
  from nltk.stem import PorterStemmer
  ps = PorterStemmer()
  ps.stem('running')                         # 'run'
  ps.stem('organization')                    # 'organ' (aggressive!)

LEMMATIZATION
  from nltk.stem import WordNetLemmatizer
  lem = WordNetLemmatizer()
  lem.lemmatize('running', 'v')              # 'run' (pass POS!)
  lem.lemmatize('better', 'a')               # 'good'

COUNTVECTORIZER (Bag of Words)
  from sklearn.feature_extraction.text import CountVectorizer
  cv = CountVectorizer(max_features=100)
  X_train = cv.fit_transform(train_docs)     # fit + transform
  X_test  = cv.transform(test_docs)          # transform only!
  cv.get_feature_names_out()                 # vocabulary

TFIDFVECTORIZER
  from sklearn.feature_extraction.text import TfidfVectorizer
  tfidf = TfidfVectorizer(
      max_features=100,
      ngram_range=(1, 2),     # unigrams + bigrams
      min_df=2,               # ignore rare words
      max_df=0.9,             # ignore very common words
      stop_words='english',   # sklearn stopwords
      sublinear_tf=True       # log TF
  )

TF-IDF FORMULA
  TF  = count of word in document
  IDF = log(total_docs / docs_with_word)
  TF-IDF = TF * IDF
  High TF-IDF = frequent in this doc, rare overall

SPARSE MATRICES
  X.toarray()              # convert to dense numpy array
  X.shape                  # (n_docs, n_features)
  X.nnz                    # number of non-zero entries

GOLDEN RULES
  1. fit on train, transform on test
  2. Use bigrams to capture negation ("not good")
  3. Stemming = fast but crude, Lemmatization = slower but accurate
  4. TF-IDF usually beats raw counts for classification
============================================
"""
print(cheat_sheet)

---
**Next up: Day 12 — Embeddings and Transformers**

You'll learn about Word2Vec, GloVe, the attention mechanism, and how BERT changed NLP forever. See you there!